In [1]:
import time

In [2]:
start_notebook = time.time()

In [3]:
import warnings
warnings.filterwarnings("ignore")

# 1. Parameters

In [4]:
name_dataset = 'DB_Pedia'
name_model = 'claude-4.5'
mode = 'zero'
seed = 3
part = 2

In [5]:
path_open = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/03.Inference/{name_dataset}/post17/df_test_{part}.csv'

In [6]:
path_save = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/03.Inference/{name_dataset}/post18/df_test_{part}.csv'

In [7]:
path_credentials = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/credentials/ANTHROPIC_API_KEY.json'

# 2. Load Environment

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
!pip install -q anthropic

In [10]:
import json
import requests
import pandas as pd
from anthropic import Anthropic

In [11]:
with open(path_credentials, "r") as f:
    credentials = json.load(f)

In [12]:
ANTHROPIC_API_KEY = credentials["ANTHROPIC_API_KEY"]

In [13]:
client = Anthropic(api_key=ANTHROPIC_API_KEY)

# 3. Functions

In [14]:
def zero_shot_prompt(text):

    intro = (
        "Classify the topic of the following text from the DBpedia Ontology dataset.\n"
        "Respond ONLY with a single digit (0–13) according to the category:\n"
        "0 = Company\n"
        "1 = EducationalInstitution\n"
        "2 = Artist\n"
        "3 = Athlete\n"
        "4 = OfficeHolder\n"
        "5 = MeanOfTransportation\n"
        "6 = Building\n"
        "7 = NaturalPlace\n"
        "8 = Village\n"
        "9 = Animal\n"
        "10 = Plant\n"
        "11 = Album\n"
        "12 = Film\n"
        "13 = WrittenWork\n"
        "Do not include any explanation or text, only the digit."
    )

    target = f'Text: "{text}"\nLabel:'
    return intro + target

In [15]:
def predict_label(text, prompt):

    try:

        start_time = time.perf_counter()
        first_token_time = None
        output_text = ""
        usage = None

        with client.messages.stream(
            model="claude-sonnet-4-5-20250929",
            max_tokens=16,
            temperature=0.0,
            messages=[{"role": "user", "content": prompt}],
        ) as stream:
            for event in stream:
                if event.type == "content_block_delta":
                    delta_text = getattr(event.delta, "text", None)
                    if delta_text:
                        if first_token_time is None:
                            first_token_time = time.perf_counter()
                        output_text += delta_text

            final_msg = stream.get_final_message()
            usage = getattr(final_msg, "usage", None)

        end_time = time.perf_counter()

        return {
            "prediction": output_text.strip(),
            "latency_ms": (end_time - start_time) * 1000,
            "ttft_ms": ((first_token_time - start_time) * 1000) if first_token_time else None,
            "input_tokens": getattr(usage, "input_tokens", None) if usage else None,
            "output_tokens": getattr(usage, "output_tokens", None) if usage else None,
        }

    except:

        return {
            "prediction": '-1',
            "latency_ms": '-',
            "ttft_ms": '-',
            "input_tokens": '-',
            "output_tokens": '-',
        }

# 4. Load Dataset

In [16]:
df = pd.read_csv(path_open)

In [17]:
df.shape

(1400, 61)

In [18]:
pred_label = []
pred_latency = []
pred_ttft = []
pred_input_tokens = []
pred_output_tokens = []

In [19]:
for i in range(len(df)):

  text = df['text'].iloc[i]
  prompt = zero_shot_prompt(text)
  output = predict_label(text, prompt)

  pred_label.append(int(output['prediction']))
  pred_latency.append(output['latency_ms'])
  pred_ttft.append(output['ttft_ms'])
  pred_input_tokens.append(output['input_tokens'])
  pred_output_tokens.append(output['output_tokens'])

  time.sleep(1)

  if (i % 10) == 0:
    print(i)

0


10


20


30


40


...

1350


1360


1370


1380


1390


In [20]:
df[f'{name_model}-{mode}-seed-{seed}-label'] = pred_label
df[f'{name_model}-{mode}-seed-{seed}-latency'] = pred_latency
df[f'{name_model}-{mode}-seed-{seed}-ttft'] = pred_ttft
df[f'{name_model}-{mode}-seed-{seed}-input-tokens'] = pred_input_tokens
df[f'{name_model}-{mode}-seed-{seed}-output-tokens'] = pred_output_tokens

In [21]:
df[f'{name_model}-{mode}-seed-{seed}-label'].value_counts()

,count
claude-4.5-zero-seed-3-label,
8,111
9,111
11,106
3,104
2,103
13,102
10,99
5,98
4,97


In [22]:
df[f'{name_model}-{mode}-seed-{seed}-latency'].describe()

,claude-4.5-zero-seed-3-latency
count,1400.000000
mean,1200.779480
std,1265.225552
min,586.793292
25%,790.796044
50%,1195.753528
75%,1328.680276
max,45264.462449


In [23]:
df[f'{name_model}-{mode}-seed-{seed}-ttft'].describe()

,claude-4.5-zero-seed-3-ttft
count,1400.000000
mean,1136.354584
std,1268.311679
min,516.501313
25%,714.426349
50%,1141.001681
75%,1274.175280
max,45263.812001


In [24]:
df[f'{name_model}-{mode}-seed-{seed}-input-tokens'].describe()

,claude-4.5-zero-seed-3-input-tokens
count,1400.000000
mean,225.961429
std,40.942173
min,163.000000
25%,196.000000
50%,226.000000
75%,252.000000
max,834.000000


In [25]:
df[f'{name_model}-{mode}-seed-{seed}-output-tokens'].describe()

,claude-4.5-zero-seed-3-output-tokens
count,1400.0
mean,5.0
std,0.0
min,5.0
25%,5.0
50%,5.0
75%,5.0
max,5.0


# 5. Save Dataset

In [26]:
df.to_csv(path_save, index = False)

# 6. Execution Time

In [27]:
end_notebook = time.time()

In [28]:
delta_notebook = end_notebook - start_notebook
hours_notebook, rem_notebook = divmod(delta_notebook, 3600)
minutes_notebook, seconds_notebook = divmod(rem_notebook, 60)

print(f"Execution Notebook: {int(hours_notebook)}h {int(minutes_notebook)}m {seconds_notebook:.2f}s")

Execution Notebook: 0h 53m 30.71s
